# 🔍 Workshop 2 — Unsupervised Learning: Finding Groups Without Labels

**AI Frontier · Hands-on session**

In supervised learning we gave the model the answers. Here we **don't**.
The model only gets raw data and has to discover structure on its own. This is **unsupervised learning**.

We'll use **K-means clustering**, the "hello world" of unsupervised learning.

**Plan**
- **Part A:** K-means on simple 2-D points — see exactly how it works
- **Part B:** Real data — group **customers** into segments
- **Part C:** Group **cat & dog photos** without ever telling the model which is which

> ⚙️ Part C is faster on a GPU: `Runtime → Change runtime type → T4 GPU`.

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

SEED = 42

---
# Part A — How K-means works

**The idea:** pick *K* centre points, then repeat:
1. Assign every point to its nearest centre
2. Move each centre to the middle of its points

…until the centres stop moving.

We generate some 2-D points. **We know** there are 4 groups, but the model will **not** be told.

In [ ]:
X, _hidden_truth = make_blobs(n_samples=500, centers=4, cluster_std=0.9, random_state=SEED)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=15, c="gray")
plt.title("Raw data — no labels, no colours")
plt.show()

In [ ]:
# Watch K-means learn, one step at a time
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, n_iter in zip(axes, [1, 2, 3, 10]):
    km = KMeans(n_clusters=4, init="random", n_init=1, max_iter=n_iter, random_state=3).fit(X)
    ax.scatter(X[:, 0], X[:, 1], c=km.labels_, s=12, cmap="viridis")
    ax.scatter(*km.cluster_centers_.T, c="red", marker="X", s=200, edgecolor="black")
    ax.set_title(f"after {n_iter} iteration(s)")
plt.show()

### But how do we pick K?

In real data nobody tells us how many groups exist. Two common tricks:

- **Elbow method** — plot *inertia* (how spread out each cluster is) for K = 1…10; look for the "elbow" where improvement slows down.
- **Silhouette score** — how well-separated the clusters are (closer to 1 = better).

In [ ]:
ks = range(1, 11)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_) if k > 1 else np.nan)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(ks, inertias, "o-"); a1.set_title("Elbow method"); a1.set_xlabel("K"); a1.set_ylabel("inertia")
a2.plot(ks, silhouettes, "o-"); a2.set_title("Silhouette score"); a2.set_xlabel("K")
plt.show()

💬 **Discuss:** Both plots should point to **K = 4**, the true number of groups, found without any labels.

---
# Part B — Customer segmentation (real-world use case)

A shop wants to group its customers to run targeted offers. Nobody has labelled customers as "VIP" or "budget". We let K-means find the groups.

**Dataset: Mall Customers** — 200 customers with age, annual income and a spending score (1–100).

📥 **Download options**
- Original on Kaggle (free account needed): https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
- The next cell tries to download it automatically from a public GitHub mirror. If that fails, download `Mall_Customers.csv` from Kaggle and upload it with the folder icon on the left (📁).

In [ ]:
CSV = "Mall_Customers.csv"
if not os.path.exists(CSV):
    !wget -q -O Mall_Customers.csv https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv

df = pd.read_csv(CSV)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()

In [ ]:
# The original file names the columns slightly differently in some copies — normalise them
rename = {}
for c in df.columns:
    lc = c.lower()
    if "income" in lc: rename[c] = "Income"
    elif "spending" in lc: rename[c] = "Spending"
    elif lc == "age": rename[c] = "Age"
df = df.rename(columns=rename)

plt.figure(figsize=(6, 5))
plt.scatter(df["Income"], df["Spending"], s=20, c="gray")
plt.xlabel("Annual income (k$)"); plt.ylabel("Spending score (1-100)")
plt.title("Customers — can you spot groups by eye?")
plt.show()

In [ ]:
features = df[["Income", "Spending"]].values
scaled = StandardScaler().fit_transform(features)   # put both columns on the same scale

inertias = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(scaled).inertia_ for k in range(1, 11)]
plt.plot(range(1, 11), inertias, "o-"); plt.xlabel("K"); plt.ylabel("inertia"); plt.title("Elbow method")
plt.show()

In [ ]:
K = 5
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(scaled)
df["Segment"] = km.labels_

plt.figure(figsize=(7, 6))
plt.scatter(df["Income"], df["Spending"], c=df["Segment"], cmap="tab10", s=30)
centres = StandardScaler().fit(features).inverse_transform(km.cluster_centers_)
plt.scatter(centres[:, 0], centres[:, 1], c="red", marker="X", s=250, edgecolor="black")
plt.xlabel("Annual income (k$)"); plt.ylabel("Spending score (1-100)")
plt.title(f"{K} customer segments found by K-means")
plt.show()

df.groupby("Segment")[["Age", "Income", "Spending"]].mean().round(1)

💬 **Your turn:** Look at the table. Give each segment a business name.
e.g. *high income + high spending* = **"VIPs"**, *high income + low spending* = **"careful savers"**, *low income + high spending* = **"impulse buyers"**…

The model found the groups. **Humans** give them meaning.

---
# Part C — Grouping cat & dog photos with no labels

Same photos as Workshop 1, but this time **we throw away the labels**.

**Dataset:** a 2,000-photo subset of Kaggle's Dogs vs Cats. The next cell downloads the full dataset from Microsoft (~800 MB, about a minute) and copies out 1,000 cats + 1,000 dogs.
Manual link: https://www.microsoft.com/en-us/download/details.aspx?id=54765

**How it works**
1. Turn each photo into a list of 1,280 numbers (**features**) using a pre-trained MobileNetV2
2. Run K-means with K = 2 on those numbers
3. Only at the very end, peek at the real labels to see how well it did

In [ ]:
import zipfile

# Google's old "cats_and_dogs_filtered.zip" link is gone (403), so we download Microsoft's
# original Kaggle Cats vs Dogs zip and build the same small 1000 / 500 split ourselves.
URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
ZIP = "kagglecatsanddogs.zip"
OUT = "cats_and_dogs_filtered"
N_TRAIN, N_VAL = 1000, 500   # images per class

# ~800 MB, about a minute on Colab. -c resumes a partial download; the timeout + retries recover from a stalled connection.
!wget -q --show-progress -c --read-timeout=30 --tries=20 -O {ZIP} {URL}

with zipfile.ZipFile(ZIP) as z:
    for src, cls in [("Cat", "cats"), ("Dog", "dogs")]:
        names = sorted((n for n in z.namelist()
                        if n.startswith(f"PetImages/{src}/") and n.endswith(".jpg")),
                       key=lambda n: int(os.path.basename(n)[:-4]))
        kept = 0
        for n in names:
            data = z.read(n)
            if b"JFIF" not in data[:10]:      # skip the few corrupt / non-JPEG files
                continue
            split = "train" if kept < N_TRAIN else "validation"
            os.makedirs(os.path.join(OUT, split, cls), exist_ok=True)
            with open(os.path.join(OUT, split, cls, os.path.basename(n)), "wb") as f:
                f.write(data)
            kept += 1
            if kept == N_TRAIN + N_VAL:
                break

print(os.listdir(os.path.join(OUT, "train")))

In [ ]:
import tensorflow as tf
from tensorflow import keras

IMG_SIZE = (160, 160)
ds = keras.utils.image_dataset_from_directory(
    "cats_and_dogs_filtered/train", image_size=IMG_SIZE, batch_size=64,
    label_mode="int", shuffle=False)
class_names = ds.class_names
file_paths = ds.file_paths

feature_extractor = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet", pooling="avg")
preprocess = keras.applications.mobilenet_v2.preprocess_input

features, true_labels = [], []
for images, labels in ds:
    features.append(feature_extractor(preprocess(images), training=False).numpy())
    true_labels.append(labels.numpy())       # kept aside ONLY for checking at the end
features = np.concatenate(features)
true_labels = np.concatenate(true_labels)
print("Feature matrix:", features.shape)   # (2000, 1280)

In [ ]:
# Reduce 1280 numbers -> 50 with PCA (faster, less noise), then cluster
feats_50 = PCA(n_components=50, random_state=SEED).fit_transform(features)
km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(feats_50)
clusters = km.labels_
print("Images per cluster:", np.bincount(clusters))

In [ ]:
# What did each cluster collect? (still no labels used)
def show_cluster(c, n=8):
    idx = np.where(clusters == c)[0][:n]
    plt.figure(figsize=(16, 2.5))
    for i, j in enumerate(idx):
        plt.subplot(1, n, i + 1)
        plt.imshow(keras.utils.load_img(file_paths[j], target_size=IMG_SIZE))
        plt.axis("off")
    plt.suptitle(f"Cluster {c}")
    plt.show()

for c in range(2):
    show_cluster(c)

In [ ]:
# 2-D map of all 2000 photos
coords = PCA(n_components=2, random_state=SEED).fit_transform(features)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6))
a1.scatter(*coords.T, c=clusters, cmap="coolwarm", s=8); a1.set_title("Colour = cluster found by K-means")
a2.scatter(*coords.T, c=true_labels, cmap="coolwarm", s=8); a2.set_title("Colour = real label (revealed now)")
plt.show()

In [ ]:
# Finally: how well do the clusters match reality?
# Clusters have no names, so match each cluster to the label most of its images have.
correct = 0
for c in range(2):
    members = true_labels[clusters == c]
    majority = np.bincount(members).argmax()
    correct += (members == majority).sum()
    print(f"Cluster {c} -> mostly '{class_names[majority]}'  ({(members == majority).mean():.0%} pure)")
print(f"\nOverall match with real labels: {correct / len(true_labels):.1%}  — and K-means never saw a label!")

⚠️ **Honest footnote:** the MobileNetV2 feature extractor was itself trained (supervised) on ImageNet. So we're clustering *smart features*. The K-means step used zero labels, but it's standing on a model that once did. Try clustering **raw pixels** instead (Challenge 2) to see how much that matters.

## 🧪 Challenges

1. Part C: set `n_clusters=4`. What do the extra clusters capture? (Breeds? Colours? Indoor vs outdoor?)
2. Part C: cluster raw pixels instead of features — replace `features` with `np.concatenate([x.numpy().reshape(len(x), -1) for x, _ in ds])`. How does the match drop?
3. Part B: add `Age` as a third feature. Do the segments change?
4. Part A: set `cluster_std=2.5` in `make_blobs`. When does K-means start to struggle?

### ✅ Recap
- **Unsupervised learning** finds structure in data with **no labels**.
- **K-means**: assign to nearest centre → move centres → repeat.
- Choose K with the **elbow** method or **silhouette** score.
- The model finds groups; **humans interpret** what they mean.